# JAX-Gap Primitives — End-to-End Demo

A tour of the uniqx primitive surface: convolutions, linalg factorizations,
neural-network activations, distribution scoring, physics grid operators, and
graph Laplacians — every primitive routed through the uniqx gateway and verified
against numpy / scipy / networkx references.

This notebook intentionally uses small inputs (≤ 8x8) so every cell completes
in well under a second against a warm gateway.

## 1. Connect

In [ ]:
import os
import numpy as np
from uniqx import connect, submit, get, login
from uniqx.core.tracing import trace
from uniqx.core import types

endpoint = os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=endpoint)
client = connect(endpoint)
print("connected:", endpoint)


def run(module, runtime_inputs=None):
    """Submit a traced module, wait, return the parsed result payload as
    a flat numpy array."""
    job_id = submit(module, client=client, runtime_inputs=runtime_inputs or [])
    result = get(job_id, client=client, wait=True, timeout=60)
    payload = result.get("result_payload") or result.get("payload") or ""
    if isinstance(payload, (bytes, bytearray)):
        payload = payload.decode("utf-8", errors="replace")
    if "=" not in payload:
        return np.asarray([], dtype=float)
    body = payload.split("=", 1)[1]
    import re
    nums = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", body)
    return np.asarray([float(n) for n in nums], dtype=float)


## 2. NN activations — relu, gelu

In [ ]:
from uniqx.ops.primitives import nn

x = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])

@trace
def prog_relu(arr):
    return nn.relu(arr)

print("relu(x) =", run(prog_relu(x.tolist()), [f"{len(x)}xf64=" + " ".join(map(str, x.tolist()))]))
print("  ref   =", np.maximum(x, 0))


@trace
def prog_gelu(arr):
    return nn.gelu(arr)

print("gelu(x) =", run(prog_gelu(x.tolist()), [f"{len(x)}xf64=" + " ".join(map(str, x.tolist()))]))


## 3. Distribution scoring — norm.logpdf

In [ ]:
from uniqx.ops.primitives import distributions as dist

x = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])

@trace
def prog_logpdf(arr):
    return dist.norm.logpdf(arr, loc=0.0, scale=1.0)

actual = run(prog_logpdf(x.tolist()), [f"{len(x)}xf64=" + " ".join(map(str, x.tolist()))])
from scipy.stats import norm as _ref_norm
print("norm.logpdf =", actual)
print("  scipy ref =", _ref_norm.logpdf(x))


## 4. Convolution — conv1d (3-tap kernel)

In [ ]:
from uniqx.ops.primitives import conv as conv_prim
from uniqx.core.enums import Padding

# (N=1, C_in=1, L=8) input; (C_out=1, C_in=1, K=3) kernel.
x = np.arange(8.0).reshape(1, 1, 8)
k = np.array([0.25, 0.5, 0.25]).reshape(1, 1, 3)

def _bv_nd(arr):
    a = np.asarray(arr, dtype=float)
    return f"{'x'.join(str(d) for d in a.shape)}xf64=" + " ".join(map(str, a.flatten().tolist()))

@trace
def prog_conv(x_arr, k_arr):
    return conv_prim.conv1d(x_arr, k_arr, stride=1, padding=Padding.VALID, dilation=1)

actual = run(prog_conv(x.tolist(), k.tolist()), [_bv_nd(x), _bv_nd(k)])
from scipy.signal import correlate
ref = correlate(x[0, 0], k[0, 0], mode="valid")
print("conv1d =", actual)
print("  ref =", ref)


## 5. Linalg factorizations — lstsq, svd, qr, lu

In [ ]:
from uniqx.ops.primitives import linalg as la

# 4x4 SPD-ish A and rhs b.
A = np.array([
    [3.0, 1.0, 0.0, 0.5],
    [1.0, 4.0, 2.0, 0.0],
    [0.0, 2.0, 5.0, 1.0],
    [0.5, 0.0, 1.0, 6.0],
])
b = np.array([5.0, 17.0, 19.0, 22.0])

def _bv_mat(M):
    rows, cols = M.shape
    return f"{rows}x{cols}xf64=" + " ".join(map(str, M.flatten().tolist()))

def _bv_vec(v):
    return f"{len(v)}xf64=" + " ".join(map(str, v.tolist()))


@trace
def prog_lstsq(M, rhs):
    return la.lstsq(M, rhs)

x_actual = run(prog_lstsq(A.tolist(), b.tolist()), [_bv_mat(A), _bv_vec(b)])
print("lstsq x =", x_actual)
print("  numpy =", np.linalg.lstsq(A, b, rcond=None)[0])


@trace
def prog_svd_s(M):
    _u, s, _vt = la.svd(M)
    return s

s_actual = run(prog_svd_s(A.tolist()), [_bv_mat(A)])
print("svd S  =", s_actual)
print("  numpy =", np.linalg.svd(A, compute_uv=False))


## 6. Physics — kinetic-operator eigenvalue

In [ ]:
from uniqx.domains.physics.kernels import grid_laplacian

nx, ny = 6, 6
dx = 1.0 / (nx + 1)

@trace
def prog_lap():
    return grid_laplacian(nx=nx, ny=ny, nz=1, dx=dx, dy=dx)

flat = run(prog_lap())
n = nx * ny
L = flat.reshape(n, n)

# Eigenvalue check on sin(pi x) sin(pi y).
xs = np.linspace(dx, 1 - dx, nx)
ys = np.linspace(dx, 1 - dx, ny)
X, Y = np.meshgrid(xs, ys, indexing="ij")
f = (np.sin(np.pi * X) * np.sin(np.pi * Y)).flatten()
ratio = (L @ f)[np.abs(f) > 1e-6] / f[np.abs(f) > 1e-6]
print("Lf/f ratio  ≈", ratio.mean(), "  (expected ≈", -2.0 * np.pi**2, ")")


## 7. Spin chain — TFI Hamiltonian (4 spins)

In [ ]:
from uniqx.core.enums import SpinChainModel
from uniqx.domains.physics.kernels import spin_chain_hamiltonian

n_spins = 4
J, h = 1.0, 0.5

@trace
def prog_tfi():
    return spin_chain_hamiltonian(
        n_spins=n_spins, model=SpinChainModel.TFI, J=J, h_field=h, pbc=False
    )

flat = run(prog_tfi())
D = 2 ** n_spins
H = flat.reshape(D, D)
eigs = np.linalg.eigvalsh(H)
print(f"TFI ground-state energy (n={n_spins}, J={J}, h={h}):", eigs[0])
print(f"  spectrum span: [{eigs[0]:.3f}, {eigs[-1]:.3f}]")


## 8. ML — graph_laplacian

In [ ]:
from uniqx.core.enums import GraphLaplacianVariant
from uniqx.domains.ml.kernels import graph_laplacian

adj = np.array([
    [0., 1., 1., 0., 0.],
    [1., 0., 1., 1., 0.],
    [1., 1., 0., 0., 1.],
    [0., 1., 0., 0., 1.],
    [0., 0., 1., 1., 0.],
])
n = adj.shape[0]

@trace
def prog_lap(A):
    return graph_laplacian(A, n_nodes=n, variant=GraphLaplacianVariant.COMBINATORIAL)

flat = run(prog_lap(adj.tolist()), [_bv_mat(adj)])
L = flat.reshape(n, n)
ref = np.diag(adj.sum(axis=1)) - adj
print("graph_laplacian ‖actual − ref‖∞ =", np.max(np.abs(L - ref)))


## What was demonstrated

| Cell | Primitive | Reference |
|---|---|---|
| 2 | `nn.relu`, `nn.gelu` | numpy zero-clip, scipy gelu closed-form |
| 3 | `dist.norm.logpdf` | scipy.stats.norm.logpdf |
| 4 | `conv.conv1d` | scipy.signal.correlate |
| 5 | `linalg.lstsq`, `linalg.svd` | numpy.linalg |
| 6 | `grid_laplacian` | analytic eigenvalue −2π² |
| 7 | `spin_chain_hamiltonian` (TFI) | hand-built Pauli kron |
| 8 | `graph_laplacian` | D − A reference |

Complex-dtype primitives are not exercised in this notebook.